# Result Log Dashboard

Reads `results/result-log.csv` (the authoritative append-only log populated
by `eval_runner.run_eval` and `improve.infer`) and plots three views:

1. Accuracy over time, coloured by `method`.
2. Per-task leaderboard sorted by value.
3. Delta-vs-baseline with 95% CI error bars for every non-baseline run.

Regenerate the data by running any of:

```bash
uv run python -m eval_runner.run_eval --task custom_qa --method baseline
bash improve/eval.sh                          # full Part E ladder
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(context="notebook", style="whitegrid")

CSV = Path("result-log.csv")
if not CSV.exists():
    CSV = Path("../results/result-log.csv")
print(f"reading {CSV.resolve()}")

df = pd.read_csv(CSV)
# Coerce numeric columns (csv writes empty strings for missing fields).
for col in ("value", "stderr", "delta", "ci95_low", "ci95_high", "wall_s"):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df.head()

In [ ]:
(
    df.groupby(["task", "method"])
    .agg(n_runs=("run_id", "count"), last_value=("value", "last"), mean_wall_s=("wall_s", "mean"))
    .sort_values("last_value", ascending=False)
    .round(4)
)

## Accuracy over time by method

In [ ]:
per_task = df["task"].unique()
fig, axes = plt.subplots(
    nrows=len(per_task), figsize=(9, 3.2 * len(per_task)), squeeze=False, sharex=True
)
for ax, task in zip(axes[:, 0], per_task):
    sub = df[df["task"] == task].sort_values("timestamp")
    if sub.empty:
        continue
    sns.lineplot(
        data=sub, x="timestamp", y="value", hue="method", marker="o", ax=ax, errorbar=None
    )
    ax.set_title(f"{task} -- value over time")
    ax.set_ylabel("metric value")
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False)
plt.tight_layout()
plt.show()

## Per-task leaderboard (latest run per method)

In [ ]:
latest = (
    df.sort_values("timestamp")
    .groupby(["task", "method"], as_index=False)
    .last()
)
for task, sub in latest.groupby("task"):
    sub = sub.sort_values("value", ascending=True)
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(sub) + 1.5))
    ax.barh(sub["method"], sub["value"])
    ax.set_title(f"{task} -- latest value per method")
    ax.set_xlabel("value")
    for i, v in enumerate(sub["value"]):
        ax.text(v, i, f" {v:.4f}", va="center")
    plt.tight_layout()
    plt.show()

## Delta vs baseline with 95% CI

In [ ]:
with_delta = df.dropna(subset=["delta", "ci95_low", "ci95_high"]).copy()
if with_delta.empty:
    print("No runs with delta/CI yet -- run a non-baseline variant via improve.infer.")
else:
    for task, sub in with_delta.groupby("task"):
        sub = sub.sort_values("timestamp")
        fig, ax = plt.subplots(figsize=(9, 0.4 * len(sub) + 2))
        y = range(len(sub))
        err = [
            (sub["delta"] - sub["ci95_low"]).abs().to_numpy(),
            (sub["ci95_high"] - sub["delta"]).abs().to_numpy(),
        ]
        ax.errorbar(sub["delta"], list(y), xerr=err, fmt="o", capsize=4)
        ax.axvline(0, color="grey", linestyle="--", alpha=0.5)
        ax.set_yticks(list(y))
        ax.set_yticklabels(sub["method"])
        ax.set_xlabel("delta vs baseline (95% CI)")
        ax.set_title(f"{task} -- variant deltas")
        plt.tight_layout()
        plt.show()

### Notes

- Rows with `mock_backend=true` are pipeline smoke runs, not real numbers.
  Filter them out with `df[df['mock_backend'] == 'false']` when reading only
  the real-GPU tier.
- `ci95_low` / `ci95_high` come from `common.stats.paired_bootstrap` on
  per-example correctness vectors; wider CIs imply smaller `limit`.